In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns
from scipy.stats import chi2_contingency
import math

In [ ]:
url = 'https://drive.google.com/file/d/10O0PIEmKJKBwTzo-LOWG2yFYjqBOmklK/view?usp=drive_link'
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
eniac_a = pd.read_csv(path)

url = 'https://drive.google.com/file/d/1n8If-uTf75uLmgMsPdtY2Fxd9ePDL6ts/view?usp=drive_link'
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
eniac_b = pd.read_csv(path)

url = 'https://drive.google.com/file/d/1RJwDcLEnk5Gqnl_BYch6PEYzZ4NIb_hd/view?usp=drive_link'
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
eniac_c = pd.read_csv(path)

url = 'https://drive.google.com/file/d/10bpN4EkZe5PNqKW3cHXIXiATTFitnwc2/view?usp=drive_link'
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
eniac_d = pd.read_csv(path)


eniac_a.head()
eniac_b.head()
eniac_c.head()
eniac_d.head()

,Element ID,Tag name,Name,No. clicks,Visible?,Snapshot information
0,48,h1,ENIAC,285,True,Homepage Version D - red SEE DEALS • https...
1,25,div,mySidebar,305,True,created 2021-10-27 • 14 days 0 hours 34 mi...
2,4,a,Mac,274,True,NaN
3,69,a,iPhone,243,True,NaN
4,105,a,Accessories,1267,True,NaN


# Hypothesis testing: Chi-Square Test within the Eniac case study

In this notebook we perform a chi-square test with the data from the Eniac case study, applying a post-hoc correction to perform pairwise tests and find the true winner.

## 1.&nbsp;State the Null Hypothesis and the Alternative Hypothesis.

**Null Hypothesis:**
all versions have the same CTR.

**Alternative Hypothesis:**
there is a difference in the CTR for the different versions.

## 2.&nbsp; Select an appropriate significance level alpha ($\alpha$).

It was decided that a relatively high alpha was acceptable in this case


In [ ]:
alpha= 0.05

## 3.&nbsp; Collect data that is random and independent

The important pieces of information (clicks on each element of interest & visits on each page) are scattered around. Let's collect them. Where are the .csv files? 🥸

In [ ]:
eniac_a

,Element ID,Tag name,Name,No. clicks,Visible?,Snapshot information,version
0,48,h1,ENIAC,269,True,Homepage Version A - white SHOP NOW • http...,A
1,25,div,mySidebar,309,True,created 2021-09-14 • 14 days 0 hours 34 mi...,A
2,4,a,Mac,279,True,NaN,A
3,69,a,iPhone,246,True,NaN,A
4,105,a,Accessories,1235,True,NaN,A
5,36,a,Chargers & Cables,1261,False,NaN,A
6,99,a,iPhone Accessories,1226,False,NaN,A
7,68,a,Watch Accessories,1261,False,NaN,A
8,13,a,Mac Accessories,1308,False,NaN,A
9,15,a,AirTag,206,False,NaN,A


In [ ]:
button_mask= eniac_a['Name']== 'SHOP NOW'

eniac_a.loc[button_mask]

,Element ID,Tag name,Name,No. clicks,Visible?,Snapshot information,version
21,106,a,SHOP NOW,512,True,NaN,A


In [ ]:
eniac_a_clicks = eniac_a.loc[button_mask, "No. clicks"].iloc[0]

In [ ]:
eniac_a.loc[1, 'Snapshot information']

'created 2021-09-14   •   14 days 0 hours 34 mins   •   25326 visits, 23174 clicks'

In [ ]:
eniac_a.iloc[1, 5]

'created 2021-09-14   •   14 days 0 hours 34 mins   •   25326 visits, 23174 clicks'

In [ ]:
eniac_a.iloc[1, -1]

'created 2021-09-14   •   14 days 0 hours 34 mins   •   25326 visits, 23174 clicks'

In [ ]:
import re

# Put your already-loaded dataframes in a dict keyed by version
data = {'A': eniac_a, 'B': eniac_b, 'C': eniac_c, 'D': eniac_d}

def get_clicks(version):
    page_df = data[version]
    page_info = page_df['Snapshot information'][1]
    # Extract num visits from snapshot info
    visits = re.findall(pattern=r'(\d+) visits', string=page_info)[0]
    num_visits = int(visits)
    # Find element being tested
    button = page_df['Name'].isin(['SEE DEALS', 'SHOP NOW'])
    # Get clicked and didn't click
    clicked = page_df.loc[button, 'No. clicks'].iloc[0]
    didnt_click = num_visits - clicked
    return clicked, didnt_click

versions = ['A', 'B', 'C', 'D']

observed = pd.DataFrame(
    columns=versions,
    index=['Clicked', "Didn't Click"]
)

for version in versions:
    observed[version] = get_clicks(version)

observed

,A,B,C,D
Clicked,512,281,527,193
Didn't Click,24814,24466,24349,25040


In [ ]:
# Calculating statistical significance
chisq, pvalue, df, expected = chi2_contingency(observed)
# Comparing with threshold
pvalue < alpha

np.True_

## 4.&nbsp; Calculate the test result

In [ ]:
chisq, pvalue, df, expected = chi2_contingency(observed)

In [ ]:
# Don't be afraid to be wrong!
_, pvalue, _, _ = chi2_contingency(observed)
pvalue

np.float64(2.7161216607868712e-48)

In [ ]:
chisq

np.float64(224.01877488058412)

In [ ]:
pvalue

np.float64(2.7161216607868712e-48)

In [ ]:
df

3

In [ ]:
expected

array([[  382.48625502,   373.74189974,   375.69012397,   381.08172127],
       [24943.51374498, 24373.25810026, 24500.30987603, 24851.91827873]])

#### Step 4: Calculating Statistical Significance

In [ ]:
# Calculating statistical significance
chisq, pvalue, df, expected = chi2_contingency(observed)
# Comparing with threshold
pvalue < alpha

np.True_

## 5.&nbsp; Interpret the test result

In [ ]:
if pvalue < alpha:
    print("Reject the Null Hypothesis")
    print("There is a statistically significant difference between the versions.")
else:
    print("Fail to reject the Null Hypothesis")
    print("There is not enough evidence of a statistically significant difference.")

Reject the Null Hypothesis
There is a statistically significant difference between the versions.


## How do we decide who's the winner? Post-Hoc Testing

Post-hoc testing requires running a lot of tests. This increases our overall chance of getting type 1 errors, so to mitigate this we have to reduce our alpha during post-hoc testing. There are several strategies for this, but an easy one is to simply divide alpha by the number of tests being run:

In [ ]:
num_versions = len(versions)
# Getting a count of all the ways they could be paired
num_comparisons = math.comb(num_versions, 2)
# Dividing alpha by the number of comparisons
adjusted_alpha = alpha/num_comparisons

We don't just want to know which differences are statistically significant, we also want to know which versions are best. To figure this out, every time we see a statistically significant difference we'll have to calculate the click through rates for the versions to figure out which one is better. We can then eliminate the one we know is worse from the running. If we do this for all the comparisons we should end up with only the versions that have tied for first:

In [ ]:
# Calculating CTR for each version
ctrs = (observed / observed.sum()).iloc[0,:]
ctrs

,Clicked
A,0.020216
B,0.011355
C,0.021185
D,0.007649


In [ ]:
# Set of all versions (to remove losers from)
contenders = set(versions)

# Looping through pairs and computing significance
for v1 in versions:
    for v2 in versions:

        # Running new chi2 test with each pair of versions
        pair_contingency_table = observed[[v1,v2]]
        chisq, pvalue, df, expected = chi2_contingency(pair_contingency_table)

        # If statistically significant, removing loser from contenders
        if pvalue < adjusted_alpha:
            # (we only need to check 1 direction because the loop will repeat the same pair with v1 and v2 swapped)
            if ctrs.loc[v1] > ctrs.loc[v2]:
                # Removing loser from contenders
                contenders -= set(v2)
contenders

{'A', 'C'}

Alternatively, if you want to avoid comparing A to B and then B to A, you can use this code which avoids repeated pairs. It's a little harder to read for beginners though which is why the above code is provided as well.

In [31]:
contenders = set(versions)

# Looping through pairs and computing significance
for idx1 in range(num_versions-1): #(0, 1, 2)
    for idx2 in range(idx1+1, num_versions): #(1, 2, 3), (2, 3), (3)

        # Running new chi2 test with each pair of versions
        pair_contingency_table = observed.iloc[:, [idx1, idx2]]
        _, pvalue, _, _ = chi2_contingency(pair_contingency_table)

        # If statistically significant, removing loser from contenders
        if pvalue < adjusted_alpha:
            # Figuring out which is worse...
            loser = idx2 if ctrs.iloc[idx1] > ctrs.iloc[idx2] else idx1
            # and removing it from the contenders
            contenders -= set(versions[loser])

contenders

{'A', 'C'}

It looks like A and C are better than the others, but the comparison between the two of them was inconclusive. In this case we should lean toward sticking with version A. The whole reason we are being careful and checking statistical significance is so that we don't make changes to the website until we have good evidence that they will improve things. In this case the result is inconclusive whether version C could improve things so we should stick with version A.

However, before we make any conclusions, we should check the guardrail metrics to make sure the difference was not due to chance. We may even be able to break the tie if we see that one version has significantly better scores with these other metrics.

In [ ]:
num_test= 3 + 2 + 1 + 0
adjusted_alpha = alpha/ num_test

In [ ]:
num_versions= len(versions)

In [ ]:
for idx1 in range(num_versions -1):
    for idx2 in range(idx1 +1, num_versions):
        print (idx1, idx2)

0 1
0 2
0 3
1 2
1 3
2 3


In [ ]:
for version_1 in versions:
    for version_2 in version:
        print(version_1, version_2)

A Version_A
A Version_B
A Version_C
A Version_D
B Version_A
B Version_B
B Version_C
B Version_D
C Version_A
C Version_B
C Version_C
C Version_D
D Version_A
D Version_B
D Version_C
D Version_D


In [ ]:
results = pd.DataFrame({
    'Version': versions,
    'Clicks': clicks,
    'Visits': visits,
    'CTR (%)': ctr
})

results

NameError: name 'clicks' is not defined

In [ ]:
winner = results.loc[results['CTR (%)'].idxmax()]

winner

,0
Version,A
Clicks,23174
Visits,99988
CTR (%),23.176781
